In [0]:
from pyspark.sql.functions import col, explode, current_timestamp


json_path = "/Volumes/workspace/steam/steam_volume/steam_raw.json"


df_raw = (
    spark.read
    .option("multiline", "true")
    .json(json_path)
)

df_flat = (
    df_raw
    .select(explode(col("response.games")).alias("game"))
    .select(
        col("game.appid").alias("appid"),
        col("game.name").alias("name"),
        col("game.playtime_forever").alias("playtime_total"),
        col("game.playtime_windows_forever").alias("playtime_windows"),
        col("game.rtime_last_played").alias("last_played_unix"),
        col("game.has_community_visible_stats").alias("has_stats")
    )
)

df_bronze = df_flat.withColumn("ingest_ts", current_timestamp())

bronze_path = "dbfs:/Volumes/workspace/steam/steam_volume/bronze"

df_bronze.write.format("delta").mode("overwrite").save(bronze_path)



In [0]:
%sql
CREATE OR REPLACE TABLE workspace.steam.bronze_df AS
SELECT *
FROM delta.`/Volumes/workspace/steam/steam_volume/bronze`;
